In [ ]:

!git clone https://github.com/fburlacu/ds-ts-project.git

Cloning into 'ds-ts-project'...
remote: Enumerating objects: 87422, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 87422 (delta 28), reused 12 (delta 8), pack-reused 87367 (from 3)
Receiving objects: 100% (87422/87422), 1.69 GiB | 15.82 MiB/s, done.
Resolving deltas: 100% (10142/10142), done.
Updating files: 100% (87279/87279), done.


In [ ]:

!pip install numpy pandas scikit-learn matplotlib tqdm
!pip install git+https://github.com/moment-timeseries-foundation-model/moment.git

  Cloning https://github.com/moment-timeseries-foundation-model/moment.git to /tmp/pip-req-build-m0tv3j7q
  Running command git clone --filter=blob:none --quiet https://github.com/moment-timeseries-foundation-model/moment.git /tmp/pip-req-build-m0tv3j7q
  Resolved https://github.com/moment-timeseries-foundation-model/moment.git to commit 38f7310ad594100747ca2a8357e9c7ca7d323e0e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for momentfm: filename=momentfm-0.1.5-py3-none-any.whl size=34253 sha256=acd989aea45263d83218417ad034b459421cdce4e828990bd0caa7c746e66086
  Stored in directory: /tmp/pip-ephem-wheel-cache-od7qx6wh/wheels/d4/4c/8a/a636e1a7e41d9fdfa5c6dd584d6b9f7c447a252d098899a73c
Successfully built momentfm


In [ ]:

!pip install wfdb pandas numpy torch scikit-learn matplotlib seaborn transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 116.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you h

In [ ]:
from momentfm import MOMENTPipeline

model = MOMENTPipeline.from_pretrained(
     "AutonLab/MOMENT-1-small",
    model_kwargs={
        'task_name' : 'classification',
        'n_channels': 12,
        'num_class': 2,
        'freeze_encoder': True,
        'freeze_embedder': True,
        'freeze_head': False,
        'enable_gradient_checkpointing': False,
        'reduction': 'mean',
    },
)


In [ ]:
model.init()
print(model)

MOMENTPipeline(
  (normalizer): RevIN()
  (tokenizer): Patching()
  (patch_embedding): PatchEmbedding(
    (value_embedding): Linear(in_features=8, out_features=512, bias=False)
    (position_embedding): PositionalEmbedding()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (

/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


In [ ]:
import os
import ast
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR



from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
SAMPLING_RATE = 100          # 100Hz → 1000 timesteps
N_LEADS       = 12
SEQ_LEN       = 1000
RANDOM_STATE  = 42

DATA_DIR = '/content/ds-ts-project/ecg-dataset/PTBXL/'

def is_normal(scp_dict):
        for code in scp_dict:
            if code == "NORM":
                return 0
        return 1



def load_ptbxl(data_dir, sampling_rate=100):
    """Load PTB-XL signals and binary labels (NORM=1, Abnormal=0)."""
    df = pd.read_csv(os.path.join(data_dir, 'ptbxl_database.csv'), index_col='ecg_id')
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)

    df['label'] = df.scp_codes.apply(is_normal)

    print(f"Dataset size : {len(df)}")
    print(f"Normal       : {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
    print(f"Abnormal     : {(1-df['label']).sum()} ({(1-df['label'].mean())*100:.1f}%)")

    filename_col = 'filename_lr' if sampling_rate == 100 else 'filename_hr'
    signals = []
    for _, row in df.iterrows():
        path = os.path.join(data_dir, row[filename_col])
        sig, _ = wfdb.rdsamp(path)
        signals.append(sig)

    X = np.stack(signals).transpose(0, 2, 1)
    y = df['label'].values
    return X, y



def normalize(X_train, X_val, X_test):
    """Standardize per-lead using training set statistics only."""
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std  = X_train.std(axis=(0, 2),  keepdims=True) + 1e-8
    return (X_train-mean)/std, (X_val-mean)/std, (X_test-mean)/std


X, y = load_ptbxl(DATA_DIR, SAMPLING_RATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val   = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=RANDOM_STATE)

X_train, X_val, X_test = normalize(X_train, X_val, X_test)

print(f"\nSplit sizes → train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")


Dataset size : 21799
Normal       : 12285 (56.4%)
Abnormal     : 9514 (43.6%)

Split sizes → train: 15695, val: 1744, test: 4360


In [ ]:
BATCH_SIZE = 32

class PTBXLDataset(Dataset):
    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


train_loader = DataLoader(PTBXLDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PTBXLDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(PTBXLDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_X, sample_y = next(iter(train_loader))
print(f"Input shape per batch: {sample_X.shape}")
print(f"Batches per epoch → train: {len(train_loader)}, val: {len(val_loader)}")

Input shape per batch: torch.Size([32, 12, 1000])
Batches per epoch → train: 491, val: 55


In [ ]:
def train_epoch(model, device, train_dataloader, criterion, optimizer, scheduler, reduction='mean'):

    model.to(device)
    model.train()
    losses = []

    for batch_x, batch_labels in tqdm(train_dataloader, desc="Training", leave=False):
        optimizer.zero_grad()
        batch_x = batch_x.to(device).float()
        batch_labels = batch_labels.to(device)


        with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
            output = model(x_enc=batch_x, reduction=reduction)
            loss = criterion(output.logits, batch_labels)
        loss.backward()

        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    avg_loss = np.mean(losses)
    return avg_loss

In [ ]:
def evaluate_epoch(dataloader, model, criterion, device, phase='val', reduction='mean'):
    model.eval()
    model.to(device)
    total_loss, total_correct = 0, 0

    with torch.no_grad():
        for batch_x, batch_labels in dataloader:
            batch_x = batch_x.to(device).float()
            batch_labels = batch_labels.to(device)

            output = model(x_enc=batch_x, reduction=reduction)
            loss = criterion(output.logits, batch_labels)
            total_loss += loss.item()
            total_correct += (output.logits.argmax(dim=1) == batch_labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = total_correct / len(dataloader.dataset)
    return avg_loss, accuracy


In [ ]:

from tqdm import tqdm
import numpy as np

epoch = 5
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3, total_steps=epoch * len(train_loader))
device = 'cuda'

for i in tqdm(range(epoch)):
    train_loss = train_epoch(model, device, train_loader, criterion, optimizer, scheduler)
    val_loss, val_accuracy = evaluate_epoch(val_loader, model, criterion, device, phase='test')
    print(f'Epoch {i}, train loss: {train_loss}, val loss: {val_loss}, val accuracy: {val_accuracy}')

test_loss, test_accuracy = evaluate_epoch(test_loader, model, criterion, device, phase='test')
print(f'Test loss: {test_loss}, test accuracy: {test_accuracy}')

 20%|██        | 1/5 [08:07<32:28, 487.24s/it]

Epoch 0, train loss: 0.6490146585734459, val loss: 0.5600271371277896, val accuracy: 0.7723623853211009



 40%|████      | 2/5 [16:10<24:15, 485.09s/it]

Epoch 1, train loss: 0.5122803339768816, val loss: 0.4471416088667783, val accuracy: 0.8067660550458715



 60%|██████    | 3/5 [24:15<16:10, 485.04s/it]

...


 80%|████████  | 4/5 [32:20<08:04, 484.86s/it]

Epoch 3, train loss: 0.435576680539339, val loss: 0.40768530531363056, val accuracy: 0.8182339449541285



100%|██████████| 5/5 [40:23<00:00, 484.75s/it]

Epoch 4, train loss: 0.43132075236431205, val loss: 0.4058873645283959, val accuracy: 0.8211009174311926


Test loss: 0.43132772867697, test accuracy: 0.8068807339449541
